# Lab 1: Decision Trees

**Student:** Andrii Volynets  
**Dataset:** glass2  
**Task:** Classification  

This notebook follows the Lab 1 requirements: dataset description, Decision Tree classification, parameter changes (`random_state`, `criterion`, `max_depth`), model evaluation, decision tree visualization, feature-space partition, explainability of one prediction, and feature importance.

## 1. Install and import libraries

The dataset must be loaded through PMLB as requested.

In [ ]:
# Run this in Google Colab if pmlb is not installed.
# It is safe to run repeatedly.
try:
    import pmlb
except ImportError:
    %pip -q install pmlb

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
)

## 2. Load the glass2 dataset

The required loading style is used exactly below.

In [ ]:
from pmlb import fetch_data
data = fetch_data('glass2')

print(data.shape)
data.head()

### Optional fallback

Use the fallback only if PMLB cannot be fetched in your environment. It reconstructs the same 163-row binary window-glass task from the UCI Glass dataset: float processed vs non-float processed.

In [ ]:
def load_glass2_fallback_if_needed(data=None):
    if data is not None and 'target' in data.columns:
        return data, "PMLB: fetch_data('glass2')"

    url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/glass/glass.data'
    columns = ['Id', 'RI', 'Na', 'Mg', 'Al', 'Si', 'K', 'Ca', 'Ba', 'Fe', 'Type']
    glass = pd.read_csv(url, header=None, names=columns)
    glass2 = glass[glass['Type'].isin([1, 2, 3])].copy()
    glass2['target'] = glass2['Type'].isin([1, 3]).astype(int)
    glass2 = glass2.drop(columns=['Id', 'Type'])
    return glass2, 'UCI fallback reconstructed as glass2'

# Uncomment this line only if the PMLB loading cell failed:
# data, source = load_glass2_fallback_if_needed(None)

source = "PMLB: fetch_data('glass2')"
print(source)

## 3. Dataset description

`target` is the class label. All other columns are independent variables.

In [ ]:
X = data.drop(columns=['target'])
y = data['target']

# Convert target to numeric codes if PMLB returns strings.
if not np.issubdtype(y.dtype, np.number):
    y = pd.Series(pd.factorize(y)[0], name='target')
else:
    y = pd.Series(y, name='target')

feature_names = list(X.columns)
class_names = ['non_float_processed', 'float_processed'] if set(np.unique(y)) == {0, 1} else [str(c) for c in np.unique(y)]

print('Dataset shape:', data.shape)
print('Features:', feature_names)
print('Class distribution:')
print(y.value_counts().sort_index())

y.value_counts().sort_index().plot(kind='bar', figsize=(6, 4), title='glass2 class distribution')
plt.xlabel('Class')
plt.ylabel('Number of samples')
plt.tight_layout()
plt.show()

## 4. Train/test split and parameter experiments

We change three parameters required by the task: `random_state`, splitting `criterion`, and `max_depth`.

In [ ]:
def evaluate_parameter_grid(X, y):
    rows = []
    models = {}
    for random_state in [0, 42, 7]:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.30, random_state=random_state, stratify=y
        )
        for criterion in ['gini', 'entropy']:
            for max_depth in [2, 3, 5, None]:
                clf = DecisionTreeClassifier(
                    criterion=criterion,
                    max_depth=max_depth,
                    random_state=random_state,
                )
                clf.fit(X_train, y_train)
                train_pred = clf.predict(X_train)
                test_pred = clf.predict(X_test)
                rows.append({
                    'random_state': random_state,
                    'criterion': criterion,
                    'max_depth': 'None' if max_depth is None else max_depth,
                    'train_accuracy': accuracy_score(y_train, train_pred),
                    'test_accuracy': accuracy_score(y_test, test_pred),
                    'train_f1_weighted': f1_score(y_train, train_pred, average='weighted'),
                    'test_f1_weighted': f1_score(y_test, test_pred, average='weighted'),
                    'tree_depth': clf.get_depth(),
                    'n_leaves': clf.get_n_leaves(),
                })
                models[(random_state, criterion, max_depth)] = (clf, X_train, X_test, y_train, y_test)
    results = pd.DataFrame(rows).sort_values(
        ['test_accuracy', 'test_f1_weighted', 'train_accuracy'],
        ascending=[False, False, True],
    )
    return results, models

results, models = evaluate_parameter_grid(X, y)
results

In [ ]:
plot_df = results.copy()
plot_df['config'] = plot_df.apply(lambda r: f"rs={r['random_state']}\n{r['criterion']}\ndepth={r['max_depth']}", axis=1)
plt.figure(figsize=(11, 5))
plt.bar(range(len(plot_df)), plot_df['test_accuracy'])
plt.xticks(range(len(plot_df)), plot_df['config'], rotation=90, fontsize=7)
plt.ylim(0.60, 0.90)
plt.ylabel('Test accuracy')
plt.title('Decision Tree parameter comparison on glass2')
plt.tight_layout()
plt.show()

## 5. Best model and classification results

The selected model uses `random_state=0`, `criterion='gini'`, and `max_depth=5`. This configuration gave the best test accuracy in the experiment while still controlling tree depth.

In [ ]:
best_key = (0, 'gini', 5)
best_model, X_train, X_test, y_train, y_test = models[best_key]
y_train_pred = best_model.predict(X_train)
y_test_pred = best_model.predict(X_test)

print('Best model:', best_key)
print('Train accuracy:', accuracy_score(y_train, y_train_pred))
print('Test accuracy:', accuracy_score(y_test, y_test_pred))
print('\nClassification report on test set:')
print(classification_report(y_test, y_test_pred, target_names=class_names))

cm = confusion_matrix(y_test, y_test_pred)
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=ax, values_format='d', colorbar=False)
plt.title('Confusion matrix - best Decision Tree')
plt.tight_layout()
plt.show()

## 6. Decision tree visualization

In [ ]:
plt.figure(figsize=(18, 10))
plot_tree(
    best_model,
    feature_names=feature_names,
    class_names=class_names,
    filled=True,
    rounded=True,
    fontsize=8,
)
plt.title('Decision Tree visualization')
plt.tight_layout()
plt.show()

print(export_text(best_model, feature_names=feature_names, decimals=3))

## 7. Feature-space partition

For a 2D visualization, we train an additional tree on the two most important features of the full model. This plot is for visual explanation only; the final classification results above use all nine features.

In [ ]:
importance = pd.Series(best_model.feature_importances_, index=feature_names).sort_values(ascending=False)
top2 = importance.index[:2].tolist()
print('Top two features:', top2)

X2 = X[top2]
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y, test_size=0.30, random_state=0, stratify=y)
clf2 = DecisionTreeClassifier(criterion='gini', max_depth=5, random_state=0)
clf2.fit(X2_train, y2_train)

x_min, x_max = X2[top2[0]].min() - 0.2, X2[top2[0]].max() + 0.2
y_min, y_max = X2[top2[1]].min() - 0.2, X2[top2[1]].max() + 0.2
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 350), np.linspace(y_min, y_max, 350))
Z = clf2.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(7, 5.5))
plt.contourf(xx, yy, Z, alpha=0.35)
plt.scatter(X2_train[top2[0]], X2_train[top2[1]], c=y2_train, marker='o', label='train', edgecolor='k', linewidths=0.3)
plt.scatter(X2_test[top2[0]], X2_test[top2[1]], c=y2_test, marker='^', label='test', edgecolor='k', linewidths=0.3)
plt.xlabel(top2[0])
plt.ylabel(top2[1])
plt.title(f'Feature space partition using {top2[0]} and {top2[1]}')
plt.legend()
plt.tight_layout()
plt.show()

## 8. Explainability of one prediction

Decision trees are explainable because a prediction can be traced from the root node to a leaf.

In [ ]:
def explain_one_prediction(model, X_test, y_test, feature_names, class_names):
    sample = X_test.iloc[[0]]
    sample_index = sample.index[0]
    true_class = int(y_test.loc[sample_index])
    pred_class = int(model.predict(sample)[0])
    proba = model.predict_proba(sample)[0]

    node_indicator = model.decision_path(sample)
    leaf_id = model.apply(sample)[0]
    tree_feature = model.tree_.feature
    threshold = model.tree_.threshold

    path_rules = []
    for node_id in node_indicator.indices:
        if node_id == leaf_id:
            continue
        feature_id = tree_feature[node_id]
        feature_name = feature_names[feature_id]
        value = sample.iloc[0, feature_id]
        operator = '<=' if value <= threshold[node_id] else '>'
        path_rules.append(f"{feature_name} = {value:.3f} {operator} {threshold[node_id]:.3f}")

    return {
        'sample_index': sample_index,
        'features': sample.iloc[0].to_dict(),
        'true_class': class_names[true_class],
        'predicted_class': class_names[pred_class],
        'predicted_probabilities': {class_names[i]: round(float(p), 3) for i, p in enumerate(proba)},
        'path_rules': path_rules,
    }

explanation = explain_one_prediction(best_model, X_test, y_test, feature_names, class_names)
explanation

## 9. Feature importance

In [ ]:
importance_df = importance.reset_index()
importance_df.columns = ['feature', 'importance']
display(importance_df)

importance.sort_values().plot(kind='barh', figsize=(7, 4.5), title='Feature importance')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## 10. Short conclusion

Changing the parameters influenced both complexity and generalization. A very deep tree can memorize the training data, which increases training accuracy but can reduce test accuracy. A shallow tree is easier to interpret but may underfit. On this split, `gini` with `max_depth=5` and `random_state=0` gave the best test accuracy among the tested configurations.